# HRDPS Subsample

## Example Conversion

In [2]:
import os
import xarray as xr

HRDPS_path = (
    "/results/forcing/atmospheric/continental2.5/"
    "nemo_forcing/hrdps_y2023m03d01.nc"
)

name_output = "hrdps_subsampled_y2023m03d01.nc"
stride = 4

with xr.open_dataset(HRDPS_path) as ds:
    print("Original dimensions:")
    print(dict(ds.sizes))

    # x、y 方向每 4 个点取 1 个
    ds_subsampled = ds.isel(
        x=slice(None, None, stride),
        y=slice(None, None, stride),
    )

    print("Subsampled dimensions:")
    print(dict(ds_subsampled.sizes))

    # netCDF4 后端支持的 encoding 参数
    valid_encoding_keys = {
        "shuffle",
        "szip_coding",
        "chunksizes",
        "szip_pixels_per_block",
        "blosc_shuffle",
        "quantize_mode",
        "compression",
        "complevel",
        "fletcher32",
        "least_significant_digit",
        "contiguous",
        "significant_digits",
        "endian",
        "_FillValue",
        "dtype",
        "zlib",
    }

    encoding = {}

    for var_name in ds_subsampled.variables:
        original_encoding = ds[var_name].encoding

        # 只保留 netCDF4 后端接受的参数
        var_encoding = {
            key: value
            for key, value in original_encoding.items()
            if key in valid_encoding_keys
        }

        # 对 datetime 变量保留 CF 时间编码
        if "units" in original_encoding:
            var_encoding["units"] = original_encoding["units"]

        if "calendar" in original_encoding:
            var_encoding["calendar"] = original_encoding["calendar"]

        # chunksizes 是针对原始数组形状的，降采样后可能不合适
        var_encoding.pop("chunksizes", None)
        var_encoding.pop("contiguous", None)

        # 使用普通 zlib 压缩，避免继承其他压缩格式
        var_encoding.pop("compression", None)
        var_encoding.pop("szip_coding", None)
        var_encoding.pop("szip_pixels_per_block", None)
        var_encoding.pop("blosc_shuffle", None)

        if ds_subsampled[var_name].ndim > 0:
            var_encoding["zlib"] = True
            var_encoding["complevel"] = 4
            var_encoding["shuffle"] = True

        encoding[var_name] = var_encoding

    ds_subsampled.to_netcdf(
        name_output,
        mode="w",
        format="NETCDF4",
        engine="netcdf4",
        encoding=encoding,
        unlimited_dims=(
            ["time_counter"]
            if "time_counter" in ds_subsampled.dims
            else None
        ),
    )

print(f"Output saved to: {os.path.abspath(name_output)}")

Original dimensions:
{'time_counter': 24, 'y': 230, 'x': 190}
Subsampled dimensions:
{'time_counter': 24, 'y': 58, 'x': 48}
Output saved to: /home/jqiu/analysis-junqi/Analysis_Atmospheric_Forcing/Data_Conversion/Data_HRDPS_Subsample/hrdps_subsampled_y2023m03d01.nc


### Example Inspection

In [3]:
import netCDF4 as nc
import numpy as np

# Define file paths
path_HRDPS_subsampled = '/home/jqiu/analysis-junqi/Analysis_Atmospheric_Forcing/Data_Conversion/Data_HRDPS_Subsample/hrdps_subsampled_y2023m03d01.nc'
path_HRDPS = '/results/forcing/atmospheric/continental2.5/nemo_forcing/hrdps_y2023m03d01.nc'

# Target variables for comparison
TARGET_VARS = [
    'time_counter', 'x', 'y', 
    'nav_lat', 'nav_lon', 
    'atmpres', 'percentcloud', 'precip', 
    'qair', 'solar', 'tair', 
    'therm_rad', 'u_wind', 'v_wind'
]

def get_var_info(dataset, var_name):
    """Extract attributes and sample data for a single variable, returning a dictionary."""
    if var_name not in dataset.variables:
        return {"shape": "Missing", "type": "Missing", "units": "Missing", "sample": "Missing"}
    
    var_obj = dataset.variables[var_name]
    info = {
        "shape": str(var_obj.shape),
        "type": str(var_obj.datatype),
        "units": getattr(var_obj, 'units', 'None')  # Display "None" if units attribute is missing
    }
    
    # Safely read a small amount of sample data
    if np.prod(var_obj.shape) > 0:
        try:
            if len(var_obj.shape) == 1:
                sample = var_obj[:3]
            elif len(var_obj.shape) == 2:
                sample = var_obj[0, :3]
            elif len(var_obj.shape) == 3:
                sample = var_obj[0, 0, :3]
            elif len(var_obj.shape) == 4:
                sample = var_obj[0, 0, 0, :3]
            else:
                sample = var_obj[...].flatten()[:3]
            
            # Format array: truncate floats to avoid long strings, take first 3 values
            if sample.dtype.kind == 'f':
                sample_str = "[" + ", ".join([f"{val:.6g}" for val in sample]) + ", ...]"
            else:
                sample_str = "[" + ", ".join([str(val) for val in sample]) + ", ...]"
                
            info["sample"] = sample_str
        except Exception:
            info["sample"] = "Read Error"
    else:
        info["sample"] = "Empty Array"
        
    return info

def compare_nc_files_as_table(file1, file2, vars_to_check):
    print("\nOpening files for side-by-side comparison...")
    try:
        ds1 = nc.Dataset(file1, 'r')
        ds2 = nc.Dataset(file2, 'r')
    except Exception as e:
        print(f"Failed to open files: {e}")
        return
    
    # Set table column widths
    col1_w = 14  # Variable Name
    col2_w = 10  # Attribute
    col3_w = 36  # File 1 Data
    col4_w = 36  # File 2 Data
    
    # Print table header
    sep_line = "-" * (col1_w + col2_w + col3_w + col4_w + 13)
    print(sep_line)
    print(f"| {'Variable':<{col1_w}} | {'Attribute':<{col2_w}} | {'Constructed (File 1)':<{col3_w}} | {'HRDPS (File 2)':<{col4_w}} |")
    print(sep_line)
    
    # Iterate through and compare each variable
    for var in vars_to_check:
        info1 = get_var_info(ds1, var)
        info2 = get_var_info(ds2, var)
        
        # First row: Variable Name and Shape
        print(f"| {var:<{col1_w}} | {'Shape':<{col2_w}} | {info1['shape']:<{col3_w}} | {info2['shape']:<{col4_w}} |")
        # Subsequent rows: Type, Units, Sample (Variable column left blank for clean look)
        print(f"| {'':<{col1_w}} | {'Type':<{col2_w}} | {info1['type']:<{col3_w}} | {info2['type']:<{col4_w}} |")
        print(f"| {'':<{col1_w}} | {'Units':<{col2_w}} | {info1['units']:<{col3_w}} | {info2['units']:<{col4_w}} |")
        print(f"| {'':<{col1_w}} | {'Sample':<{col2_w}} | {info1['sample']:<{col3_w}} | {info2['sample']:<{col4_w}} |")
        print(sep_line)

    ds1.close()
    ds2.close()

# Execute comparison
compare_nc_files_as_table(path_HRDPS_subsampled, path_HRDPS, TARGET_VARS)


Opening files for side-by-side comparison...
-------------------------------------------------------------------------------------------------------------
| Variable       | Attribute  | Constructed (File 1)                 | HRDPS (File 2)                       |
-------------------------------------------------------------------------------------------------------------
| time_counter   | Shape      | (24,)                                | (24,)                                |
|                | Type       | float64                              | float64                              |
|                | Units      | seconds since 1970-01-01             | seconds since 1970-01-01             |
|                | Sample     | [1.67763e+09, 1.67763e+09, 1.67764e+09, ...] | [1.67763e+09, 1.67763e+09, 1.67764e+09, ...] |
-------------------------------------------------------------------------------------------------------------
| x              | Shape      | (48,)                     